In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print("project root:", root)

In [ ]:
# Same hard gate as the profiling/M0 kernels: refuse plausible-looking
# numbers from the wrong hardware. M1 only needs 1 GPU's worth of ticket
# time (scripts/train.py runs single-process here, not torchrun), but the
# project's numbers are all calibrated against 2xT4 specifically.
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers"], check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
# Inference-side: costs Kaggle weekly quota, not the 10-hour training
# budget. One 128-frame trajectory through the M0-chosen AE.
subprocess.run([
    sys.executable, "scripts/preprocess/precompute_dataset.py",
    "--tier", "m1",
    "--out-dir", "/kaggle/working/nanowm_data/m1",
    "--device", "cuda",
], check=True)

In [ ]:
# The first real GPU-hour spent against the ledger. ticket_hours=0.1 in
# the config is well under M1's 1.0 GPU-hour allocation (budget/allocation.yaml)
# -- this is a pipeline smoke run, not the full overfit budget.
#
# Three legitimate exit codes, none of them an error in this cell's sense:
#   0 = completed all configured steps
#   2 = the ticket deadline cut the run off -- the budget rule working
#   1 = an unhandled exception, most commonly NaNDetected from the watchdog
# All three still produce a ledger entry (scripts/train.py logs in `finally`),
# so the run's outcome is inspected via the ledger below rather than by
# treating any particular exit code as failure.
result = subprocess.run([
    sys.executable, "scripts/train.py",
    "--config", "configs/m1_overfit_5m.yaml",
])
print(f"train.py exit code: {result.returncode}")

In [ ]:
# Ledger entry proves the ticket/log discipline held on real hardware.
print(Path("budget/ledger.jsonl").read_text())